# MERMAID CBM structured outputs — viewer

Place this notebook **in the same folder** as the exported data products (or set `DATA_DIR` below), then run all cells.

**Requirements:** `numpy`, `matplotlib` (no other packages).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "."  # folder containing rgb_image.png, *.npy, *.json, *_hard.png
TAXONOMY_RANKS = ("kingdom", "phylum", "class", "order", "family", "genus")

In [ ]:
def load_label_json(path):
    """Load {"0": name, ...} label files without importing json."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    entries = []
    for line in text.splitlines():
        line = line.strip().rstrip(",")
        if not line or line in "{}":
            continue
        key, value = line.split(":", 1)
        idx = int(key.strip().strip('"'))
        name = value.strip().strip('"')
        entries.append((idx, name))
    entries.sort(key=lambda item: item[0])
    return [name for _, name in entries]


def rgb_float(rgb):
    out = rgb.astype(np.float64)
    if out.max() > 1.0:
        out /= 255.0
    return np.clip(out, 0.0, 1.0)


def present_label_ids(labels):
    return [int(i) for i in np.unique(labels)]


def label_id_to_colors(label_ids, cmap_name="tab20"):
    cmap = plt.get_cmap(cmap_name, max(len(label_ids), 1))
    return {lid: cmap(i)[:3] for i, lid in enumerate(label_ids)}


def labels_to_color_image(labels, id_to_color):
    color = np.zeros((*labels.shape, 3), dtype=np.float64)
    for lid, c in id_to_color.items():
        color[labels == lid] = c
    return color


def blend_overlay(rgb, overlay_rgb, overlay_weight=0.7):
    base = rgb_float(rgb)
    return np.clip((1.0 - overlay_weight) * base + overlay_weight * overlay_rgb, 0.0, 1.0)


def winning_class_probabilities(class_probs, class_hard):
    return np.take_along_axis(class_probs, class_hard[..., None], axis=2)[..., 0]


def add_present_legend(ax, label_ids, names, id_to_color):
    handles = [
        plt.Line2D(
            [0], [0], marker="s", color="w", markerfacecolor=id_to_color[lid], markersize=8, linestyle=""
        )
        for lid in label_ids
    ]
    legend_names = [names[lid] if lid < len(names) else str(lid) for lid in label_ids]
    ax.legend(
        handles, legend_names, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=7, frameon=False
    )


def show_label_map(ax, labels, title, names=None):
    present_ids = present_label_ids(labels)
    id_to_color = label_id_to_colors(present_ids)
    color_img = labels_to_color_image(labels, id_to_color)
    ax.imshow(color_img, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
    if names is not None:
        add_present_legend(ax, present_ids, names, id_to_color)


def probability_heatmap_rgb(prob_map, cmap_name="viridis"):
    cmap = plt.get_cmap(cmap_name)
    return cmap(np.clip(prob_map, 0.0, 1.0))[..., :3]

In [ ]:
rgb = plt.imread(f"{DATA_DIR}/rgb_image.png")
if rgb.dtype != np.uint8:
    rgb = np.clip(rgb, 0, 1)

class_probs = np.load(f"{DATA_DIR}/mermaid_classification_probabilities.npy")
class_hard = class_probs.argmax(axis=2).astype(np.int64)
class_names = load_label_json(f"{DATA_DIR}/mermaid_classification.json")

taxonomy = {}
for rank in TAXONOMY_RANKS:
    probs = np.load(f"{DATA_DIR}/taxonomy_{rank}_probabilities.npy")
    taxonomy[rank] = {
        "probs": probs,
        "hard": probs.argmax(axis=2).astype(np.int64),
        "names": load_label_json(f"{DATA_DIR}/taxonomy_{rank}.json"),
    }

multihot_probs = np.load(f"{DATA_DIR}/multi_hot_probabilities.npy")
multihot_names = load_label_json(f"{DATA_DIR}/multi_hot.json")

print("RGB:", rgb.shape, rgb.dtype)
print("MERMAID classes:", class_probs.shape, "hard", class_hard.shape)
print("Multi-hot:", multihot_probs.shape)

In [ ]:
present_class_ids = present_label_ids(class_hard)
class_id_to_color = label_id_to_colors(present_class_ids)
hard_colors = labels_to_color_image(class_hard, class_id_to_color)
win_probs = winning_class_probabilities(class_probs, class_hard)
prob_weighted_colors = hard_colors * win_probs[..., None]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(rgb)
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].imshow(blend_overlay(rgb, hard_colors, overlay_weight=0.7))
axes[1].set_title("0.3 × image + 0.7 × hard classification")
axes[1].axis("off")
add_present_legend(axes[1], present_class_ids, class_names, class_id_to_color)

axes[2].imshow(0.3 * rgb_float(rgb) + 0.7 * prob_weighted_colors)
axes[2].set_title("0.3 × image + 0.7 × p(class) × hard colors")
axes[2].axis("off")

plt.suptitle("MERMAID classification", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()
for ax, rank in zip(axes, TAXONOMY_RANKS):
    info = taxonomy[rank]
    show_label_map(ax, info["hard"], f"{rank} (hard)", info["names"])
plt.suptitle("Taxonomy — hard one-hot per rank", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
mean_probs = multihot_probs.mean(axis=(0, 1))
top_k = min(9, len(multihot_names))
top_idx = np.argsort(mean_probs)[::-1][:top_k]

ncols = 3
nrows = int(np.ceil(top_k / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, ch in zip(axes, top_idx):
    heat_rgb = probability_heatmap_rgb(multihot_probs[..., ch])
    ax.imshow(0.3 * rgb_float(rgb) + 0.7 * heat_rgb)
    ax.set_title(multihot_names[ch], fontsize=9)
    ax.axis("off")
for ax in axes[top_k:]:
    ax.axis("off")
plt.suptitle("Multi-hot concept probability maps", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Click any pixel on the RGB image to inspect predictions at that location.
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb)
ax.set_title("Click a pixel to inspect predictions")
ax.axis("off")
report = ax.text(
    0.02,
    0.98,
    "",
    transform=ax.transAxes,
    va="top",
    fontsize=8,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
)


def on_click(event):
    if event.inaxes is not ax or event.xdata is None or event.ydata is None:
        return
    y, x = int(round(event.ydata)), int(round(event.xdata))
    if not (0 <= y < class_hard.shape[0] and 0 <= x < class_hard.shape[1]):
        return
    cls_id = int(class_hard[y, x])
    cls_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
    lines = [f"Pixel ({x}, {y})", f"MERMAID: {cls_name} ({class_probs[y, x, cls_id]:.2f})"]
    for rank in TAXONOMY_RANKS:
        info = taxonomy[rank]
        rid = int(info["hard"][y, x])
        rname = info["names"][rid] if rid < len(info["names"]) else str(rid)
        lines.append(f"{rank}: {rname} ({info['probs'][y, x, rid]:.2f})")
    active = [multihot_names[i] for i, p in enumerate(multihot_probs[y, x]) if p >= 0.5]
    lines.append("Multi-hot: " + (", ".join(active) if active else "(none)"))
    report.set_text("\n".join(lines))
    fig.canvas.draw_idle()


fig.canvas.mpl_connect("button_press_event", on_click)
plt.show()